# Gold — Date Dimension (Kimball)

Generates a comprehensive daily date spine from 2022-01-01 to 2030-12-31.
Follows Kimball Data Warehouse Toolkit conventions for date dimension design.

**Output:** `gold.dim_date` (Delta table)  
**Grain:** One row per calendar day  
**Surrogate key:** `date_key` — YYYYMMDD integer  

| Column | Type | Description |
|---|---|---|
| `date_key` | int | Surrogate key — YYYYMMDD (e.g. 20240101) |
| `full_date` | date | Calendar date — natural key |
| `day_of_week_num` | int | Day of week — 1 (Sunday) to 7 (Saturday) |
| `day_name` | string | Full day name (e.g. Monday) |
| `day_of_month` | int | Day number within the month (1–31) |
| `day_of_year` | int | Day number within the year (1–366) |
| `week_of_year` | int | ISO week number (1–53) |
| `month_num` | int | Month number (1–12) |
| `month_name` | string | Full month name (e.g. January) |
| `month_short_name` | string | Abbreviated month name (e.g. Jan) |
| `quarter_num` | int | Quarter number (1–4) |
| `quarter_label` | string | Year and quarter (e.g. 2024 Q1) |
| `year_num` | int | Calendar year (e.g. 2024) |
| `year_month_key` | int | YYYYMM integer — joins to fact month_key |
| `is_weekday` | boolean | True if Monday–Friday |
| `is_weekend` | boolean | True if Saturday or Sunday |
| `is_month_start` | boolean | True if first day of the month |
| `is_month_end` | boolean | True if last day of the month |
| `is_quarter_start` | boolean | True if first day of January, April, July, or October |
| `is_quarter_end` | boolean | True if last day of March, June, September, or December |
| `is_year_start` | boolean | True if 1 January |
| `is_year_end` | boolean | True if 31 December |

In [ ]:
df_dim_date = spark.sql("""
    SELECT
        YEAR(date) * 10000 + MONTH(date) * 100 + DAY(date)  AS date_key,
        date                                                  AS full_date,
        DAYOFWEEK(date)                                       AS day_of_week_num,
        DATE_FORMAT(date, 'EEEE')                             AS day_name,
        DAY(date)                                             AS day_of_month,
        DAYOFYEAR(date)                                       AS day_of_year,
        WEEKOFYEAR(date)                                      AS week_of_year,
        MONTH(date)                                           AS month_num,
        DATE_FORMAT(date, 'MMMM')                             AS month_name,
        DATE_FORMAT(date, 'MMM')                              AS month_short_name,
        QUARTER(date)                                         AS quarter_num,
        CONCAT(YEAR(date), ' Q', QUARTER(date))               AS quarter_label,
        YEAR(date)                                            AS year_num,
        YEAR(date) * 100 + MONTH(date)                        AS year_month_key,
        CASE WHEN DAYOFWEEK(date) NOT IN (1,7)
             THEN TRUE ELSE FALSE END                         AS is_weekday,
        CASE WHEN DAYOFWEEK(date) IN (1,7)
             THEN TRUE ELSE FALSE END                         AS is_weekend,
        CASE WHEN DAY(date) = 1
             THEN TRUE ELSE FALSE END                         AS is_month_start,
        CASE WHEN date = LAST_DAY(date)
             THEN TRUE ELSE FALSE END                         AS is_month_end,
        CASE WHEN DAY(date) = 1 AND MONTH(date) IN (1,4,7,10)
             THEN TRUE ELSE FALSE END                         AS is_quarter_start,
        CASE WHEN date = LAST_DAY(date) AND MONTH(date) IN (3,6,9,12)
             THEN TRUE ELSE FALSE END                         AS is_quarter_end,
        CASE WHEN DAYOFYEAR(date) = 1
             THEN TRUE ELSE FALSE END                         AS is_year_start,
        CASE WHEN MONTH(date) = 12 AND DAY(date) = 31
             THEN TRUE ELSE FALSE END                         AS is_year_end,
        CURRENT_TIMESTAMP()                                   AS refreshed_at
    FROM (
        SELECT EXPLODE(SEQUENCE(
            DATE('2022-01-01'),
            DATE('2030-12-31'),
            INTERVAL 1 DAY
        )) AS date
    )
""")

if df_dim_date.isEmpty():
    raise ValueError("[gold_dim_date] Date spine generated no rows — halting.")

df_dim_date.createOrReplaceTempView("v_dim_date")

if not spark.catalog.tableExists("gold.dim_date"):
    df_dim_date.write.format("delta").saveAsTable("gold.dim_date")
else:
    spark.sql("""
        MERGE INTO gold.dim_date AS target
        USING v_dim_date AS source
        ON target.date_key = source.date_key
        WHEN MATCHED THEN UPDATE SET
            target.full_date        = source.full_date,
            target.day_of_week_num  = source.day_of_week_num,
            target.day_name         = source.day_name,
            target.day_of_month     = source.day_of_month,
            target.day_of_year      = source.day_of_year,
            target.week_of_year     = source.week_of_year,
            target.month_num        = source.month_num,
            target.month_name       = source.month_name,
            target.month_short_name = source.month_short_name,
            target.quarter_num      = source.quarter_num,
            target.quarter_label    = source.quarter_label,
            target.year_num         = source.year_num,
            target.year_month_key   = source.year_month_key,
            target.is_weekday       = source.is_weekday,
            target.is_weekend       = source.is_weekend,
            target.is_month_start   = source.is_month_start,
            target.is_month_end     = source.is_month_end,
            target.is_quarter_start = source.is_quarter_start,
            target.is_quarter_end   = source.is_quarter_end,
            target.is_year_start    = source.is_year_start,
            target.is_year_end      = source.is_year_end,
            target.refreshed_at     = source.refreshed_at
        WHEN NOT MATCHED THEN INSERT (
            date_key, full_date, day_of_week_num, day_name, day_of_month, day_of_year,
            week_of_year, month_num, month_name, month_short_name, quarter_num, quarter_label,
            year_num, year_month_key, is_weekday, is_weekend, is_month_start, is_month_end,
            is_quarter_start, is_quarter_end, is_year_start, is_year_end, refreshed_at
        )
        VALUES (
            source.date_key, source.full_date, source.day_of_week_num, source.day_name,
            source.day_of_month, source.day_of_year, source.week_of_year, source.month_num,
            source.month_name, source.month_short_name, source.quarter_num, source.quarter_label,
            source.year_num, source.year_month_key, source.is_weekday, source.is_weekend,
            source.is_month_start, source.is_month_end, source.is_quarter_start,
            source.is_quarter_end, source.is_year_start, source.is_year_end, source.refreshed_at
        )
    """)

print(f"Saved to gold.dim_date - {spark.table('gold.dim_date').count()} rows")